In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime

# Função para importar dados
def extract_market_data(tickers, benchmark, start_date, end_date):
    
    print(f"Baixando dados de {start_date} até {end_date}...")
    all_tickers = tickers + [benchmark]
    data = yf.download(all_tickers, start=start_date, end=end_date, progress=False)["Close"]
    
    if isinstance(data.columns, pd.MultiIndex):
        data.xs("Close", axis=1, level=0)
    
    data = data.dropna()
    return data

# Função para calcular risco/retorno
def calculate_return_risk(df, benchmark_ticker, risk_free_rate_annual=0.1440):
    returns_df = df.pct_change().dropna()
    cumulative_returns = (1 + returns_df).cumprod() - 1
    metrics = []
    risk_free_daily = (1 + risk_free_rate_annual)** (1/252) - 1

    for ticker in returns_df.columns:
        daily_mean = returns_df[ticker].mean()
        daily_std = returns_df[ticker].std()
        annual_volatility = daily_std * np.sqrt(252)
        annual_return = (1 + daily_mean) ** 252 - 1
        sharpe_ratio = (annual_return - risk_free_rate_annual) / annual_volatility
        cum_ret = (1 + returns_df[ticker]).cumprod()
        running_max = cum_ret.cummax()
        drawdown = (cum_ret / running_max) - 1
        max_drawdown = drawdown.min()

        metrics.append({
            "Ativo": ticker,
            "Retorno_Acumulado_%": round(cumulative_returns[ticker].iloc[-1] * 100, 2),
            "Volatilidade_Anual_%": round(annual_volatility * 100, 2),
            "Sharpe_Ratio": round(sharpe_ratio, 2),
            "Max_Drawdown_%": round(max_drawdown * 100, 2),
            "Tipo": "Benchmark" if ticker == benchmark_ticker else "Carteira"
        })
    
    metrics_df = pd.DataFrame(metrics)
    return returns_df, cumulative_returns, metrics_df

# Função principal
def main():
    portfolio_tickers =  ["VALE3.SA", "ITUB4.SA", "PETR4.SA", "BBAS3.SA", "BBDC4.SA", 
                          "SBSP3.SA", "B3SA3.SA", "ITSA4.SA", "ABEV3.SA", "WEGE3.SA"]
    benchmark = "^BVSP"
    start_date = "2021-01-01"
    end_date = datetime.now().strftime("%Y-%m-%d")
    
    prices_df = extract_market_data(portfolio_tickers, benchmark, start_date, end_date)
    returns_df, cum_returns_df, metrics_df = calculate_return_risk(prices_df, benchmark)

    returns_long = returns_df.reset_index().melt(id_vars="Date", var_name="Ativo", value_name="Retorno_Diario")
    returns_long["Date"] = pd.to_datetime(returns_long["Date"]).dt.date
    cum_returns_long = cum_returns_df.reset_index().melt(id_vars="Date", var_name="Ativo", value_name="Retorno_Acumulado")
    cum_returns_long["Date"] = pd.to_datetime(cum_returns_long["Date"]).dt.date

    returns_long.to_csv("../data/fato_retornos_diarios.csv", index=False)
    cum_returns_long.to_csv("../data/fato_retornos_acumulados.csv", index=False)
    metrics_df.to_csv("../data/dim_metricas_risco.csv", index=False)

    print("Pipeline Concluído com Sucesso. Arquivos Exportados!")

if __name__ == "__main__":
    main()

Baixando dados de 2021-01-01 até 2026-06-08...
Pipeline Concluído com Sucesso. Arquivos Exportados!
